# Feedback-Implied Score Pilot

Development-only pilot: infer what scores existing human feedback implies, separately for SW, UA, and HA. These are secondary inferred labels, **not** expert reference scores. The packet includes the reflection, rubric, and human feedback; it excludes generated G1/G2/G3 outputs, condition labels, and gold segment annotations.

This notebook defaults to preflight only. The LLM run cell requires an explicit approval switch because it sends reflection text and human feedback to Gemini and, with tracing enabled by config, LangSmith. Raw outputs and evidence quotes are written only under approved `PREBI_DATA_ROOT`.

## 1. Load the configured development cohort

The cell below joins R1 documents from the test split to their human-feedback records. It prints only counts and validates stable segments, candidate consistency, and non-empty feedback; no text, IDs, or labels are displayed.

In [1]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from reflection_assessment_feedback import ReflectionDocument, ReflectionSegment
from reflection_assessment_feedback.experiment_config import load_experiment_config
from reflection_assessment_feedback.prompt_registry import resolve_prompt
from reflection_assessment_feedback.rubric import RUBRIC_ARTIFACT

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
EXPERIMENT_CONFIG = load_experiment_config()
DATASET_CONFIG = EXPERIMENT_CONFIG["dataset"]
GENERATION_CONFIG = EXPERIMENT_CONFIG["generation"]
PROMPTS_CONFIG = EXPERIMENT_CONFIG["prompts"]
PILOT_CONFIG = EXPERIMENT_CONFIG["feedback_implied_score_pilot"]
SPLIT_PATH = (
    PROJECT_ROOT
    / EXPERIMENT_CONFIG["paths"]["provisional_split_directory"]
    / f"provisional_{DATASET_CONFIG['prediction_split']}_modeling_wide.csv"
)
FEEDBACK_PATH = (PROJECT_ROOT / EXPERIMENT_CONFIG["paths"]["human_feedback"]).resolve()
SCORING_PROMPT = resolve_prompt(
    PROMPTS_CONFIG["feedback_implied_score_llm_id"],
    PROMPTS_CONFIG["feedback_implied_score_llm_version"],
)
assert SCORING_PROMPT.status == "development_draft"
assert RUBRIC_ARTIFACT.version == GENERATION_CONFIG["expected_rubric_version"]

feedback_rows = pd.read_csv(
    FEEDBACK_PATH,
    usecols=["essay_name", "document_id", "feedback_text"],
    dtype={"essay_name": str, "document_id": str, "feedback_text": str},
)
feedback_rows["essay_name"] = feedback_rows["essay_name"].str.strip().str.upper()
feedback_rows["document_id"] = feedback_rows["document_id"].str.strip()
feedback_rows["feedback_text"] = feedback_rows["feedback_text"].fillna("")

split_rows = pd.read_csv(
    SPLIT_PATH,
    usecols=["document_id", "segment_id", "text", "segment_order"],
    dtype={"document_id": str, "segment_id": str},
)
split_rows["document_id"] = split_rows["document_id"].str.strip()
split_rows["segment_id"] = split_rows["segment_id"].str.strip()
split_rows["text"] = split_rows["text"].fillna("").astype(str)
split_rows["segment_order"] = pd.to_numeric(split_rows["segment_order"], errors="raise").astype(int)
split_document_ids = set(split_rows["document_id"].dropna())
feedback_ids = set(
    feedback_rows.loc[
        feedback_rows["essay_name"] == DATASET_CONFIG["development_essay_name"],
        "document_id",
    ]
)
PILOT_DOCUMENT_IDS = sorted(split_document_ids & feedback_ids)
if len(PILOT_DOCUMENT_IDS) != 6:
    raise ValueError("Expected exactly six R1 development documents in the configured test split.")

PILOT_CASES = []
for document_id in PILOT_DOCUMENT_IDS:
    feedback_match = feedback_rows.loc[feedback_rows["document_id"] == document_id]
    if len(feedback_match) != 1:
        raise ValueError("Each pilot document must map to exactly one human-feedback record.")
    human_feedback = feedback_match.iloc[0]["feedback_text"]
    if not human_feedback.strip():
        raise ValueError("A pilot document has empty human feedback.")
    document_rows = split_rows.loc[split_rows["document_id"] == document_id].copy()
    consistency = document_rows.groupby("segment_id").agg(
        text_variants=("text", "nunique"),
        order_variants=("segment_order", "nunique"),
    )
    if (consistency > 1).any().any():
        raise ValueError("Candidate copies disagree on segment text or source order.")
    document_rows = document_rows.drop_duplicates("segment_id").sort_values("segment_order")
    document = ReflectionDocument(
        document_id=document_id,
        segments=[
            ReflectionSegment(
                segment_id=row.segment_id,
                text=row.text,
                order=row.segment_order,
            )
            for row in document_rows.itertuples(index=False)
        ],
    )
    PILOT_CASES.append(
        {
            "document": document,
            "human_feedback": human_feedback,
            "human_feedback_id": f"human-feedback-{document_id}",
        }
    )

print(
    f"Preflight passed: {len(PILOT_CASES)} R1 documents, "
    f"{sum(len(case['document'].segments) for case in PILOT_CASES)} segments, "
    f"{len(PILOT_CASES) * 3} dimension decisions expected. Text and IDs withheld."
)

Preflight passed: 6 R1 documents, 282 segments, 18 dimension decisions expected. Text and IDs withheld.


## 2. Run LLM inference (explicit approval required)

This cell sends the reflection, rubric, and human feedback to Gemini; tracing can also retain them in LangSmith. It makes one request per document, uses the shared cross-notebook rate limiter, skips exact matching saved results, and stops rather than overwriting conflicting records. Leave approval false until this specific data flow is authorized.

In [2]:
import hashlib
import json
import os
import tempfile
from pathlib import Path

from dotenv import dotenv_values, load_dotenv
from reflection_assessment_feedback.experiment_config import (
    experiment_config_sha256,
    load_experiment_config,
)
from reflection_assessment_feedback.feedback_implied_scoring import (
    build_feedback_score_packet,
    feedback_score_packet_sha256,
    infer_feedback_implied_score,
)
from reflection_assessment_feedback.rate_limit import reserve_request_slot

APPROVE_EXTERNAL_PROCESSING = True
assert APPROVE_EXTERNAL_PROCESSING, (
    "Set APPROVE_EXTERNAL_PROCESSING = True only after approving Gemini and LangSmith "
    "processing of R1 reflections and human feedback."
)

load_dotenv(PROJECT_ROOT / ".env", override=False)
assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), (
    "Configure GOOGLE_API_KEY or GEMINI_API_KEY in the approved local environment."
)
ENABLE_LANGSMITH_TRACING = GENERATION_CONFIG["enable_langsmith_tracing"]
if ENABLE_LANGSMITH_TRACING:
    assert os.getenv("LANGSMITH_API_KEY"), "Configure LANGSMITH_API_KEY in local .env."
    assert os.getenv("LANGSMITH_PROJECT"), "Configure LANGSMITH_PROJECT in local .env."
    assert os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com").rstrip("/") == (
        "https://eu.api.smith.langchain.com"
    ), "Tracing is restricted to the LangSmith EU endpoint."

PROTECTED_ROOT_VALUE = os.getenv("PREBI_DATA_ROOT")
if not PROTECTED_ROOT_VALUE:
    raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage.")
PROTECTED_ROOT = Path(PROTECTED_ROOT_VALUE).expanduser().resolve()
if PROTECTED_ROOT == PROJECT_ROOT or PROJECT_ROOT in PROTECTED_ROOT.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")
OUTPUT_DIRECTORY = PROTECTED_ROOT / "feedback-implied-score" / "llm-runs"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True, mode=0o700)
if not OUTPUT_DIRECTORY.resolve().is_relative_to(PROTECTED_ROOT):
    raise ValueError("Score-run storage must resolve within PREBI_DATA_ROOT.")
RATE_LIMIT_STATE = PROTECTED_ROOT / ".rate_limits" / "gemini.state"

from langchain_google_genai import ChatGoogleGenerativeAI

CHAT_MODEL = ChatGoogleGenerativeAI(
    model=GENERATION_CONFIG["model_name"],
    temperature=GENERATION_CONFIG["temperature"],
)

PILOT_RUNS = []
for case in PILOT_CASES:
    document_id = case["document"].document_id
    human_feedback_id = case["human_feedback_id"]
    packet_seed = "|".join(
        (
            document_id,
            human_feedback_id,
            SCORING_PROMPT.sha256,
            RUBRIC_ARTIFACT.sha256,
        )
    )
    packet_id = "score-packet-" + hashlib.sha256(packet_seed.encode("utf-8")).hexdigest()[:20]
    packet = build_feedback_score_packet(
        packet_id=packet_id,
        document=case["document"],
        rubric=RUBRIC_ARTIFACT.rubric,
        human_feedback=case["human_feedback"],
    )
    input_packet_sha256 = feedback_score_packet_sha256(packet)
    output_path = OUTPUT_DIRECTORY / (
        f"score-{document_id}-{SCORING_PROMPT.version}-r1.json"
    )
    if output_path.is_file():
        existing_record = json.loads(output_path.read_text(encoding="utf-8"))
        if (
            existing_record.get("input_packet_sha256") != input_packet_sha256
            or existing_record.get("prompt_artifact_sha256") != SCORING_PROMPT.sha256
            or existing_record.get("rubric_artifact_sha256") != RUBRIC_ARTIFACT.sha256
            or existing_record.get("experiment_config_sha256") != experiment_config_sha256()
        ):
            raise ValueError("Existing score record conflicts with the current packet or versions.")
        PILOT_RUNS.append(existing_record)
        continue

    reserve_request_slot(
        RATE_LIMIT_STATE,
        GENERATION_CONFIG["minimum_request_interval_seconds"],
    )
    score_record = infer_feedback_implied_score(
        model=CHAT_MODEL,
        document=case["document"],
        rubric=RUBRIC_ARTIFACT.rubric,
        human_feedback=case["human_feedback"],
        human_feedback_id=human_feedback_id,
        document_id=document_id,
        enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
        packet_id=packet_id,
    )
    if score_record["input_packet_sha256"] != input_packet_sha256:
        raise ValueError("Scorer input packet hash differs from the preflight packet.")
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=OUTPUT_DIRECTORY,
            prefix=".feedback-score-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(score_record, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write("\n")
        os.chmod(temporary_path, 0o600)
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
    PILOT_RUNS.append(score_record)

if len(PILOT_RUNS) != len(PILOT_CASES):
    raise ValueError("Not every feedback-implied scoring packet has a saved result.")
print(
    f"Pilot complete: {len(PILOT_RUNS)} score-inference records; "
    f"prompt={SCORING_PROMPT.artifact_id}@{SCORING_PROMPT.version}; "
    f"tracing={'on' if ENABLE_LANGSMITH_TRACING else 'off'}. Text and scores withheld."
)

/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/c

Pilot complete: 6 score-inference records; prompt=feedback_implied_score_llm_inference@0.2.0; tracing=on. Text and scores withheld.
